In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

In [ ]:
S = 100 #spot price
K = 90 #strike price
T = 1 #time till expiration in years
r = 0.04 #risk free rate
q = 0 #continious dividend yield
sigma = 0.4 #annualized implied volatility

d_1 = lambda S, K, r, q, s, T: (np.log(S / K) + (r - q + (s ** 2) / 2) * T) / (s * np.sqrt(T))

In [ ]:
def safety(x):
    return np.where(x > 0, x, 1e-9)

# Time decay: $\Theta$

In [ ]:
def theta(S,K,r,q,sigma,T,daily=True):
    # making sure T ≠ 0 and s ≠ 0
    T_safe, sigma_safe = safety(T), safety(sigma)

    # calculating d1 and d2
    d1 = d_1(S,K,r,q,sigma_safe,T_safe)
    d2 = d1 - sigma_safe * np.sqrt(T_safe)   

    # calculating call and put
    term1 = -(S * sigma_safe * np.exp(-q * T_safe) * stats.norm.pdf(d1)) / (2 * np.sqrt(T_safe))


    call = (term1 
                    - r * K * np.exp(-r * T_safe) * stats.norm.cdf(d2) 
                    + q * S * np.exp(-q * T_safe) * stats.norm.cdf(d1))
    put = (term1 
                    + r * K * np.exp(-r * T_safe) * stats.norm.cdf(-d2) 
                    - q * S * np.exp(-q * T_safe) * stats.norm.cdf(-d1))

    # checking if T > 0 otherwise returns 0
    call = np.where(T > 0, call, 0.0)
    put = np.where(T > 0, put, 0.0)

    # returns daily if daily = True
    if daily:
        return call/365, put/365
    else:
        return call, put

# Underlying price sensitivity: $\Delta$

In [ ]:
def delta(S,K,r,q,sigma,T):
    # making sure T ≠ 0 and s ≠ 0
    T_safe, sigma_safe = safety(T), safety(sigma)

    # calculating d1 and terms for call and put
    d1 = d_1(S,K,r,q,sigma_safe,T_safe)

    term1 = np.exp(-q * T_safe)
    N_d1 = stats.norm.cdf(d1)

    # calculating call and put
    call = term1 * N_d1
    put = term1 * (N_d1 - 1)

    return call, put

# Convexity / $\Delta$ sensitivity: $\Gamma$

In [ ]:
def gamma(S,K,r,q,sigma,T):
    # making sure T ≠ 0, s ≠ 0, S ≠ 0
    T_safe, sigma_safe, S_safe = safety(T), safety(sigma), S_safe(S)
    
    # calculating d1
    d1 = d_1(S_safe,K,r,q,sigma_safe,T_safe)

    # calculating and returning gamma 
    numerator = (np.exp(-q * T_safe) * stats.norm.pdf(d1))
    denominator = (S_safe * sigma_safe * np.sqrt(T_safe))
    gamma_val = numerator/denominator

    return  np.where(T > 0, gamma_val, 0.0) 

# Volatility Sensitivity: $\nu$

In [ ]:
def vega(S,K,r,q,sigma,T,scaled=True):
    # making sure T ≠ 0 and s ≠ 0
    T_safe, sigma_safe = safety(T), safety(sigma)

    # calculating vega
    d1 = d_1(S,K,r,q,sigma_safe,T_safe)
    vega_val = S * np.exp(-q * T) * stats.norm.pdf(d1) * np.sqrt(T)

    # scaling vega if scaling
    output = vega_val / 100 if scaled else vega_val

    return np.where((T > 0) & (s > 0), output, 0.0)

# Interest rate sensitivity: $\rho$

In [ ]:
def rho(S,K,r,q,sigma,T,scaled=True):
    # making sure T ≠ 0 and s ≠ 0
    T_safe, sigma_safe = safety(T), safety(sigma)
    
    d2 = d_1(S,K,r,q,sigma_safe,T_safe) - sigma_safe * np.sqrt(T_safe)
    term1 = K * T_safe * np.exp(-r * T_safe)

    call = term1 * stats.norm.cdf(d2)
    put = -term1 * stats.norm.cdf(-d2)

    # scaling rho if scaling
    if scaled:
        call = call / 100
        put = put / 100

    return np.where(T > 0, call, 0.0), np.where(T > 0, put, 0.0)